In [53]:
import pandas as pd
import numpy as np
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package stopwords to C:\Users\YASHUB
[nltk_data]     HAYAT\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to C:\Users\YASHUB
[nltk_data]     HAYAT\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [55]:
data = pd.read_csv(r"D:\data_science_4achievers\dataset_project2\Resume.csv")

In [57]:
job_descriptions = {
    "HR": """Human Resources professional responsible for recruitment, onboarding, 
    and employee relations. Manages talent acquisition, payroll, performance appraisals, 
    and training programs. Proficient in HRMS software, labor laws, and conflict resolution. 
    Experience with employee engagement, policy development, and organizational development. 
    Strong communication and interpersonal skills required.""",

    "DESIGNER": """Creative Designer responsible for developing visual concepts and designs 
    for digital and print media. Proficient in Adobe Photoshop, Illustrator, InDesign, 
    Figma and CorelDraw. Experience in UI/UX design, typography, color theory, and branding. 
    Creates logos, banners, social media graphics, and marketing materials. 
    Strong portfolio and attention to detail required.""",

    "INFORMATION-TECHNOLOGY": """IT Professional responsible for software development, network management,
    and technical support. Proficient in programming languages Python, Java,
    JavaScript, C++, and SQL. Experience with databases, cloud computing,
    cybersecurity, and system administration.
    Skills include: software development, web development, mobile development,
    networking, database management, cloud services AWS Azure GCP,
    DevOps, agile, scrum, SDLC, testing, debugging, API development,
    microservices, docker, kubernetes, git, linux, windows server,
    technical support, IT infrastructure, system analysis, data structures,
    algorithms, machine learning, artificial intelligence, deep learning.
    Tools: VS Code, Eclipse, GitHub, Jenkins, JIRA, Postman, MySQL,
    MongoDB, PostgreSQL, Oracle, Redis, Elasticsearch..""",

    "TEACHER": """Experienced Teacher responsible for developing lesson plans and delivering 
    quality education to students. Strong knowledge of subject matter, curriculum development, 
    and classroom management. Experience with e-learning platforms, student assessment, 
    and parent communication. Proficient in creating engaging educational content and 
    fostering a positive learning environment. Patience and strong communication skills required.""",

    "ADVOCATE": """Advocate professional responsible for providing support and assistance 
    to patients, victims, and communities in need. Experience in patient 
    advocacy, healthcare administration, and human resource management.
    Strong knowledge of healthcare systems, medical records, and 
    community resources.
    Skills include: patient advocacy, crisis intervention, risk assessment,
    safety planning, supportive counseling, case management, community 
    resources, electronic medical record EMR, epic, healthcare compliance,
    HIPAA compliance, phlebotomy, lab procedures, blood bank, venipuncture,
    domestic violence advocacy, bilingual communication, client caseload,
    healthcare administration, human resources, medical terminology,
    clinical procedures, patient care, health education, social work.
    Tools: Epic, EMR systems, Microsoft Office, Excel, Access.
    Strong communication, empathy and interpersonal skills required.""",

    "BUSINESS-DEVELOPMENT": """Business Development Manager responsible for identifying growth 
    opportunities, building client relationships, and driving revenue. Experience in market 
    research, lead generation, sales strategies, and partnership development. Proficient in 
    CRM tools, negotiation, and business planning. Strong knowledge of industry trends, 
    competitive analysis, and strategic planning. Excellent communication and networking skills required.""",

    "HEALTHCARE": """Healthcare Professional responsible for patient care, diagnosis, and treatment. 
    Strong knowledge of medical procedures, clinical practices, and patient management. 
    Experience with electronic health records, medical equipment, and healthcare regulations. 
    Proficient in patient assessment, medication administration, and health monitoring. 
    Compassionate with strong attention to detail and ability to work under pressure.""",

    "FITNESS": """Fitness Trainer responsible for designing workout programs and guiding clients 
    towards health goals. Strong knowledge of exercise science, nutrition, and injury prevention. 
    Experience in personal training, group fitness classes, and fitness assessments. 
    Proficient in strength training, cardio workouts, yoga, and rehabilitation exercises. 
    Motivational with strong communication and coaching skills required.""",

    "AGRICULTURE": """Agriculture Professional responsible for crop management, soil analysis, 
    and farm operations. Strong knowledge of agricultural practices, irrigation systems, 
    and pest control. Experience with modern farming techniques, agricultural machinery, 
    and supply chain management. Proficient in crop planning, yield optimization, and 
    sustainable farming practices. Knowledge of government agricultural policies required.""",

    "BPO": """BPO Professional responsible for handling customer queries, data processing, 
    and back office operations. Strong communication skills in handling inbound and outbound 
    calls, emails, and chat support. Experience with CRM software, ticketing systems, 
    and customer service processes. Proficient in data entry, quality analysis, and 
    process improvement. Ability to work in shifts and meet performance targets required.""",

    "SALES": """Sales Professional responsible for generating leads, closing deals, and 
    achieving revenue targets. Strong knowledge of sales techniques, customer relationship 
    management, and product knowledge. Experience in B2B and B2C sales, cold calling, 
    and client presentations. Proficient in CRM tools, sales forecasting, and negotiation. 
    Target driven with excellent communication and persuasion skills required.""",

    "CONSULTANT": """Business Consultant responsible for analyzing business problems and 
    providing strategic solutions. Strong knowledge of business processes, data analysis, 
    and project management. Experience in client management, process improvement, and 
    change management. Proficient in tools like Excel, PowerPoint, Tableau, and SQL. 
    Excellent analytical thinking and problem solving skills required.""",

    "DIGITAL-MEDIA": """Digital Media Specialist responsible for managing online content, 
    social media campaigns, and digital marketing strategies. Strong knowledge of SEO, SEM, 
    Google Analytics, and social media platforms. Experience in content creation, email 
    marketing, and paid advertising campaigns. Proficient in tools like Hootsuite, 
    Canva, and WordPress. Creative with strong analytical and communication skills required.""",

    "AUTOMOBILE": """Automobile Engineer responsible for vehicle design, maintenance, and 
    repair operations. Strong knowledge of automotive systems, engine mechanics, and 
    electrical systems. Experience with vehicle diagnostics, repair tools, and 
    automobile manufacturing processes. Proficient in CAD software, quality control, 
    and safety standards. Attention to detail and strong technical skills required.""",

    "CHEF": """Professional Chef responsible for menu planning, food preparation, and 
    kitchen management. Strong knowledge of culinary techniques, food safety, and 
    nutrition. Experience in various cuisines, kitchen equipment operation, and 
    inventory management. Proficient in recipe development, plating presentation, 
    and cost control. Creativity and ability to work under pressure required.""",

    "FINANCE": """Finance Professional responsible for financial planning, budgeting, 
    and investment analysis. Strong knowledge of accounting principles, financial modeling, 
    and risk management. Experience with financial statements, tax planning, and 
    regulatory compliance. Proficient in Excel, SAP, and financial software. 
    Strong analytical skills and attention to detail required.""",

    "APPAREL": """Apparel Professional responsible for fashion design, garment production, 
    and merchandise planning. Strong knowledge of textile materials, fashion trends, 
    and garment construction. Experience in pattern making, quality control, and 
    supply chain management. Proficient in design software like Adobe Illustrator 
    and CAD tools. Creative with strong attention to detail required.""",

    "ENGINEERING": """Engineering Professional responsible for design, development, and 
    implementation of technical solutions. Strong knowledge of engineering principles, 
    CAD software, and project management. Experience in product design, testing, 
    quality assurance, and technical documentation. Proficient in AutoCAD, MATLAB, 
    and simulation tools. Analytical mindset and problem solving skills required.""",

    "ACCOUNTANT": """Accountant responsible for managing financial records, preparing 
    tax returns, and ensuring regulatory compliance. Strong knowledge of accounting 
    principles, auditing, and financial reporting. Experience with Tally, QuickBooks, 
    SAP, and MS Excel. Proficient in budgeting, accounts payable and receivable, 
    and bank reconciliation. Attention to detail and strong numerical skills required.""",

    "CONSTRUCTION": """Construction Professional responsible for project planning, site 
    management, and execution of construction projects. Strong knowledge of civil 
    engineering, building codes, and safety regulations. Experience with AutoCAD, 
    project scheduling, and cost estimation. Proficient in managing contractors, 
    materials procurement, and quality control. Leadership and problem solving skills required.""",

    "PUBLIC-RELATIONS": """Public Relations Specialist responsible for managing brand image, 
    media relations, and corporate communications. Strong knowledge of PR strategies, 
    press release writing, and crisis management. Experience in media outreach, event 
    management, and stakeholder communication. Proficient in social media, content 
    creation, and reputation management. Excellent written and verbal communication required.""",

    "BANKING": """Banking Professional responsible for managing financial transactions, 
    customer accounts, and banking operations. Strong knowledge of banking regulations, 
    loan processing, and risk assessment. Experience with core banking software, 
    financial products, and customer relationship management. Proficient in KYC, 
    AML compliance, and financial reporting. Attention to detail and integrity required.""",

    "ARTS": """Arts Professional responsible for creating visual and performing art works 
    across various mediums. Strong knowledge of art history, design principles, and 
    creative techniques. Experience in painting, sculpture, photography, digital art, 
    or performing arts. Proficient in creative tools and art software. 
    Creative thinking, originality, and strong aesthetic sense required.""",

    "AVIATION": """Aviation Professional responsible for aircraft operations, navigation, 
    and flight safety. Strong knowledge of aviation regulations, flight procedures, 
    and aircraft systems. Experience with flight planning, air traffic communication, 
    and emergency procedures. Proficient in navigation tools, aviation software, 
    and safety protocols. Excellent decision making and situational awareness required."""
}

print("Total job descriptions created:", len(job_descriptions))
print("Categories:", list(job_descriptions.keys()))

Total job descriptions created: 24
Categories: ['HR', 'DESIGNER', 'INFORMATION-TECHNOLOGY', 'TEACHER', 'ADVOCATE', 'BUSINESS-DEVELOPMENT', 'HEALTHCARE', 'FITNESS', 'AGRICULTURE', 'BPO', 'SALES', 'CONSULTANT', 'DIGITAL-MEDIA', 'AUTOMOBILE', 'CHEF', 'FINANCE', 'APPAREL', 'ENGINEERING', 'ACCOUNTANT', 'CONSTRUCTION', 'PUBLIC-RELATIONS', 'BANKING', 'ARTS', 'AVIATION']


In [59]:
def clean_resume(text):

    text = str(text)

    # remove links
    text = re.sub(r'http\S+|www\S+|linkedin\S+|github\S+', ' ', text)

    # remove dates like Jan 2015, March 2012
    text = re.sub(
        r'\b(jan|january|feb|february|mar|march|apr|april|may|jun|june|jul|july|aug|august|sep|september|oct|october|nov|november|dec|december)\s+\d{4}\b',
        ' ',
        text,
        flags=re.IGNORECASE
    )

    # remove special characters
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)

    # remove extra spaces/newlines
    text = re.sub(r'\s+', ' ', text)

    # lowercase
    text = text.lower().strip()

    return text

data["Resume_clean"] = data["Resume_str"].apply(clean_resume) 

In [61]:
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

def lemmatize_text(text):
    tokens = text.split()
    tokens = [lemmatizer.lemmatize(w) for w in tokens if w not in stop_words]
    return ' '.join(tokens)

data["Resume_clean"] = data["Resume_clean"].apply(lemmatize_text)

In [63]:

print("Resumes preprocessed successfully!")
print("Total resumes:", len(data))

Resumes preprocessed successfully!
Total resumes: 2484


In [65]:
cleaned_job_des = {}
for role , description in job_descriptions.items():
    clean_text = clean_resume(description)
    lemmitized = lemmatize_text(clean_text)
    cleaned_job_des[role] = lemmitized

print("Job descriptions preprocessed successfully!")
    

Job descriptions preprocessed successfully!


In [67]:
# Convert to lists
resume_list = data["Resume_clean"].tolist()
job_desc_list = list(cleaned_job_des.values())
job_roles = list(cleaned_job_des.keys())

# Build common vocabulary from both
tfidf = TfidfVectorizer()
tfidf.fit(resume_list + job_desc_list)   # learn from both ✅

# Transform separately
resume_vectors  = tfidf.transform(resume_list)
job_desc_vectors = tfidf.transform(job_desc_list)

print("TF-IDF vectorization done!")
print("Resume matrix shape:", resume_vectors.shape)
print("Job desc matrix shape:", job_desc_vectors.shape)

TF-IDF vectorization done!
Resume matrix shape: (2484, 33430)
Job desc matrix shape: (24, 33430)


In [69]:
def get_top3_jobs(resume_index):
    
    # Get similarity scores between this resume and all job descriptions
    similarity_scores = cosine_similarity(
        resume_vectors[resume_index],
        job_desc_vectors
    ).flatten()
    
    # Get top 3 job indices
    top3_indices = similarity_scores.argsort()[::-1][:3]
    
    print(f"\nResume {resume_index} - Actual Category: {data['Category'].iloc[resume_index]}")
    print("Top 3 Matching Job Roles:")
    print("-" * 40)
    for rank, idx in enumerate(top3_indices, 1):
        print(f"{rank}. {job_roles[idx]} → Score: {similarity_scores[idx]:.4f}")

# Test on sample resumes
for i in [0, 100, 200, 300, 400]:
    get_top3_jobs(i)


Resume 0 - Actual Category: HR
Top 3 Matching Job Roles:
----------------------------------------
1. HR → Score: 0.1551
2. ADVOCATE → Score: 0.1243
3. DIGITAL-MEDIA → Score: 0.1017

Resume 100 - Actual Category: HR
Top 3 Matching Job Roles:
----------------------------------------
1. HR → Score: 0.0909
2. ADVOCATE → Score: 0.0766
3. HEALTHCARE → Score: 0.0645

Resume 200 - Actual Category: DESIGNER
Top 3 Matching Job Roles:
----------------------------------------
1. DESIGNER → Score: 0.1678
2. ARTS → Score: 0.1222
3. PUBLIC-RELATIONS → Score: 0.0701

Resume 300 - Actual Category: INFORMATION-TECHNOLOGY
Top 3 Matching Job Roles:
----------------------------------------
1. INFORMATION-TECHNOLOGY → Score: 0.1012
2. AUTOMOBILE → Score: 0.0732
3. BUSINESS-DEVELOPMENT → Score: 0.0647

Resume 400 - Actual Category: TEACHER
Top 3 Matching Job Roles:
----------------------------------------
1. TEACHER → Score: 0.1334
2. ENGINEERING → Score: 0.0222
3. ARTS → Score: 0.0207


In [73]:
# Test on more resumes
for i in [0, 50, 100, 150, 200, 250, 300, 350, 400, 450, 500, 550, 600, 650, 700]:
    get_top3_jobs(i)


Resume 0 - Actual Category: HR
Top 3 Matching Job Roles:
----------------------------------------
1. HR → Score: 0.1551
2. ADVOCATE → Score: 0.1243
3. DIGITAL-MEDIA → Score: 0.1017

Resume 50 - Actual Category: HR
Top 3 Matching Job Roles:
----------------------------------------
1. HR → Score: 0.1813
2. ACCOUNTANT → Score: 0.0816
3. ADVOCATE → Score: 0.0699

Resume 100 - Actual Category: HR
Top 3 Matching Job Roles:
----------------------------------------
1. HR → Score: 0.0909
2. ADVOCATE → Score: 0.0766
3. HEALTHCARE → Score: 0.0645

Resume 150 - Actual Category: DESIGNER
Top 3 Matching Job Roles:
----------------------------------------
1. DESIGNER → Score: 0.1570
2. CONSULTANT → Score: 0.0869
3. CONSTRUCTION → Score: 0.0867

Resume 200 - Actual Category: DESIGNER
Top 3 Matching Job Roles:
----------------------------------------
1. DESIGNER → Score: 0.1678
2. ARTS → Score: 0.1222
3. PUBLIC-RELATIONS → Score: 0.0701

Resume 250 - Actual Category: INFORMATION-TECHNOLOGY
Top 3 Match

In [71]:
# Check what ADVOCATE resumes actually contain
for i in [450, 500, 550]:
    print(f"\nResume {i}:")
    print(data["Resume_clean"].iloc[i][:500])  # first 500 characters
    print("="*50)


Resume 450:
patient advocate summary seeking opportunity management hr department professional experience education allow make immediate contribution integral part progressive organization education training healthcare administration human resource herzing university online city state unitted state bachelor science management human resource management kaplan university city state united state business administration management kaplan university city state wfhm reverse mentoring senior management msta busine

Resume 500:
bilingual domestic violence advocate skill word program including excel access epic electronic medical record database effort outcome eto working knowledge sa r experience bilingual domestic violence advocate company name city state provided advocacy service appropriate client need caseload approximately client person phone contact advocacy service included crisis intervention risk assessment safety planning supportive counseling assistance accessing community resource

In [81]:
import pickle

with open("tfidf2.pkl", "wb") as f2:
    pickle.dump(tfidf,f2)

with open("job_desc_vectors.pkl", "wb") as f2:
    pickle.dump(job_desc_vectors, f2)

with open("job_roles.pkl", "wb") as f2:
    pickle.dump(job_roles, f2)

In [83]:
with open("tfidf2.pkl", "rb") as f2:
    loaded_tfidf2 = pickle.load(f2)

with open("job_desc_vectors.pkl", "rb") as f2:
    loaded_job_desc_vectors = pickle.load(f2)

with open("job_roles.pkl", "rb") as f2:
    loaded_job_roles = pickle.load(f2)

print("All files saved and loaded successfully!")
print("Total job roles loaded:", len(loaded_job_roles))

All files saved and loaded successfully!
Total job roles loaded: 24


In [89]:
def predict_function_top3(res_index):
    resume_text1 = data["Resume_clean"].iloc[res_index]
    cleaned2 = clean_resume(resume_text1)
    lemmatized = lemmatize_text(cleaned2)
    vectorized = loaded_tfidf2.transform([lemmatized])
    similarity_scores = cosine_similarity(
        vectorized,
        loaded_job_desc_vectors
    ).flatten()
    top3_indices1 = similarity_scores.argsort()[::-1][:3]
    print(f"\nResume{res_index} - Actual Category: {data["Category"].iloc[res_index]}")
    print("Top 3 Matching Roles:")
    print("-" * 40)
    for rank, ind in enumerate(top3_indices1, 1):
        print(f"{rank}. {loaded_job_roles[ind]} -> Score: {similarity_scores[ind]:.4f}")


predict_function_top3(450)



Resume450 - Actual Category: ADVOCATE
Top 3 Matching Roles:
----------------------------------------
1. ADVOCATE -> Score: 0.1556
2. HEALTHCARE -> Score: 0.1521
3. BPO -> Score: 0.0653


In [91]:
for i in [0, 100, 200, 300, 400, 450, 500, 550, 600, 700]:
    predict_function_top3(i)


Resume0 - Actual Category: HR
Top 3 Matching Roles:
----------------------------------------
1. HR -> Score: 0.1556
2. ADVOCATE -> Score: 0.1247
3. DIGITAL-MEDIA -> Score: 0.1020

Resume100 - Actual Category: HR
Top 3 Matching Roles:
----------------------------------------
1. HR -> Score: 0.0911
2. ADVOCATE -> Score: 0.0767
3. HEALTHCARE -> Score: 0.0647

Resume200 - Actual Category: DESIGNER
Top 3 Matching Roles:
----------------------------------------
1. DESIGNER -> Score: 0.1681
2. ARTS -> Score: 0.1225
3. PUBLIC-RELATIONS -> Score: 0.0703

Resume300 - Actual Category: INFORMATION-TECHNOLOGY
Top 3 Matching Roles:
----------------------------------------
1. INFORMATION-TECHNOLOGY -> Score: 0.1013
2. AUTOMOBILE -> Score: 0.0733
3. BUSINESS-DEVELOPMENT -> Score: 0.0647

Resume400 - Actual Category: TEACHER
Top 3 Matching Roles:
----------------------------------------
1. TEACHER -> Score: 0.1334
2. ENGINEERING -> Score: 0.0222
3. ARTS -> Score: 0.0207

Resume450 - Actual Category: A